# 00 — Hello, LLM

Goal of M0: send the same Thai and English prompts to a **local** model (Typhoon2-8B via Ollama) and a **hosted** model (Gemini, free tier), and see four core ideas in action:

1. **Tokens** and the **context window**
2. **System vs user prompt**
3. **Temperature**
4. **Why LLMs hallucinate** (and why this project needs RAG)

Prerequisites: `brew services start ollama`, the model pulled, and `GEMINI_API_KEY` in `.env` (see `.env.example`).

In [1]:
import os

import ollama
from google import genai
from google.genai import types

from thairag.config import load_env

load_env("../.env")  # puts GEMINI_API_KEY etc. into os.environ; the key never appears in the notebook

OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "scb10x/llama3.1-typhoon2-8b-instruct")
GEMINI_MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.5-flash")
gemini = genai.Client()  # reads GEMINI_API_KEY from the environment
print(OLLAMA_MODEL, "|", GEMINI_MODEL)

scb10x/llama3.1-typhoon2-8b-instruct | gemini-3.5-flash


## Two small helpers

Both return `(answer, input_tokens, output_tokens)` so we can compare the two models like for like.

- **Local:** the `ollama` package calls the Ollama server on `localhost:11434`. Nothing leaves the laptop.
- **Hosted:** the `google-genai` SDK calls Google's API over the internet. On the free tier Google may use the prompts to improve its products — fine here because Thai law is public text, but never send private data this way.

In [2]:
def ask_local(prompt, system=None, temperature=0.0):
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": prompt})
    r = ollama.chat(model=OLLAMA_MODEL, messages=messages, options={"temperature": temperature})
    return r.message.content, r.prompt_eval_count, r.eval_count


def ask_gemini(prompt, system=None, temperature=0.0):
    r = gemini.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(system_instruction=system, temperature=temperature),
    )
    u = r.usage_metadata
    return r.text, u.prompt_token_count, u.candidates_token_count


def show(name, result):
    text, n_in, n_out = result
    print(f"--- {name}  (in={n_in} tokens, out={n_out} tokens)\n{text}\n")

## 1. Tokens and the context window

**Tokens.** An LLM doesn't read letters or words; it reads *tokens* — chunks from a fixed vocabulary (~100k–250k entries) learned from data. Common English words are usually one token; rarer words and most Thai text get split into several pieces. Analogy: like one-hot encoding a categorical feature, except the "categories" are sub-word pieces and the text is first chopped into them.

**Context window.** The model can only "see" a fixed number of tokens at once (prompt + answer). Think of it as the maximum number of rows a query can scan: anything beyond is simply not there. Typhoon2-8B supports 128K tokens, **but Ollama loads it with only 8,192 by default** (`ollama ps` shows this) to save RAM; longer prompts are silently truncated unless we raise `options={"num_ctx": ...}` (M6). Gemini Flash: ~1M. Everything is priced and rate-limited in tokens, so **Thai costing more tokens than English matters for cost and for how many law sections fit in a prompt.**

Below: the same sentence in Thai and English, counted with Gemini's tokenizer (`count_tokens` is free and doesn't generate anything).

In [3]:
pair = {
    "th": "ผู้ขายต้องส่งมอบทรัพย์สินที่ขายให้แก่ผู้ซื้อ",
    "en": "The seller must deliver the property sold to the buyer.",
}
for lang, sentence in pair.items():
    n = gemini.models.count_tokens(model=GEMINI_MODEL, contents=sentence).total_tokens
    print(f"{lang}: {len(sentence):3d} characters -> {n:3d} tokens   ({len(sentence) / n:.1f} chars/token)")

th:  44 characters ->  14 tokens   (3.1 chars/token)
en:  55 characters ->  12 tokens   (4.6 chars/token)


## 2. System vs user prompt — one Thai and one English question to both models

- **System prompt** = standing instructions for the whole conversation (role, rules, format). Like the `WHERE` clause that applies to every query in a view.
- **User prompt** = this turn's actual question.

The model sees both as tokens; the system prompt just gets more weight because models are trained to obey it. Later (M6) our system prompt will say *"answer only from the given law sections and cite them"*.

In [4]:
SYSTEM = "You are a concise assistant. Answer in the same language as the question, in at most 3 sentences."
Q_TH = "สัญญาซื้อขายตามกฎหมายไทยคืออะไร"
Q_EN = "What is a contract of sale under Thai law?"

show("Typhoon (local) — Thai", ask_local(Q_TH, SYSTEM))
show("Typhoon (local) — English", ask_local(Q_EN, SYSTEM))
show("Gemini (hosted) — Thai", ask_gemini(Q_TH, SYSTEM))
show("Gemini (hosted) — English", ask_gemini(Q_EN, SYSTEM))

--- Typhoon (local) — Thai  (in=48 tokens, out=98 tokens)
สัญญาซื้อขายตามกฎหมายไทยคือสัญญาที่ทำขึ้นระหว่างผู้ซื้อและผู้ขายเพื่อการโอนกรรมสิทธิ์ในทรัพย์สิน โดยต้องมีการตกลงราคาที่ชัดเจนและเป็นธรรม สัญญานี้ต้องมีการลงนามและอาจมีการจดทะเบียนเพื่อให้มีผลทางกฎหมายที่สมบูรณ์.



--- Typhoon (local) — English  (in=46 tokens, out=120 tokens)
ในสัญญาซื้อขายตามกฎหมายไทย หมายถึง ข้อตกลงระหว่างผู้ซื้อและผู้ขายเกี่ยวกับการโอนกรรมสิทธิ์ในทรัพย์สิน โดยผู้ขายต้องส่งมอบทรัพย์สินให้แก่ผู้ซื้อ และผู้ซื้อต้องชำระเงินตามที่ตกลงกัน สัญญานี้ต้องมีการแสดงเจตนาอย่างชัดเจนจากทั้งสองฝ่ายเพื่อให้มีผลทางกฎหมาย.



--- Gemini (hosted) — Thai  (in=33 tokens, out=150 tokens)
สัญญาซื้อขายตามประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 453 คือ สัญญาซึ่งบุคคลฝ่ายหนึ่งเรียกว่า "ผู้ขาย" โอนกรรมสิทธิ์ในทรัพย์สินให้แก่บุคคลอีกฝ่ายหนึ่งเรียกว่า "ผู้ซื้อ" และผู้ซื้อตกลงว่าจะชำระราคาทรัพย์สินนั้นให้แก่ผู้ขาย สัญญานี้มีลักษณะเป็นสัญญาต่างตอบแทนที่เกิดสิทธิและหน้าที่แก่ทั้งสองฝ่ายทันทีเมื่อมีการตกลงกัน อย่างไรก็ตาม การซื้อขายทรัพย์สินบางประเภท เช่น อสังหาริมทรัพย์ กฎหมายบังคับให้ต้องทำเป็นหนังสือและจดทะเบียนต่อพนักงานเจ้าหน้าที่จึงจะสมบูรณ์ตามกฎหมาย



--- Gemini (hosted) — English  (in=34 tokens, out=102 tokens)
Under Section 453 of the Thai Civil and Commercial Code, a contract of sale is a legal agreement whereby a seller transfers ownership of property to a buyer, who agrees to pay a price for it. For certain types of property, such as real estate or specific movable assets, the law requires the contract to be in writing and registered with the relevant authorities to be valid. Once validly formed, it creates legally binding obligations for the seller to deliver the property and the buyer to pay the agreed price.



Same question, but now the system prompt asks for a different style. Only the system prompt changes:

In [5]:
show("Typhoon — 'explain to a 10-year-old'",
     ask_local(Q_TH, "Explain like I'm 10 years old, in Thai, in 2 sentences."))

--- Typhoon — 'explain to a 10-year-old'  (in=44 tokens, out=74 tokens)
สัญญาซื้อขายตามกฎหมายไทยคือเอกสารที่บันทึกการซื้อขายสินค้าหรือบริการระหว่างผู้ซื้อและผู้ขาย เพื่อให้ทั้งสองฝ่ายเข้าใจและยอมรับเงื่อนไขการซื้อขายอย่างชัดเจน



## 3. Temperature

At each step the model outputs a **probability distribution over the next token** (a softmax, like a multiclass classifier with ~100k classes). Then it *samples* one token.

- **Temperature 0:** always take the most likely token (argmax) → the same answer every time. Use this for evaluation and for legal answers.
- **Temperature 1:** sample from the distribution as is → varied wording, sometimes different content.

Statistics analogy: temperature divides the logits before the softmax; low T sharpens the distribution towards its mode, high T flattens it towards uniform.

Same open-ended question, 3 runs at each temperature (local model, so it's free):

In [6]:
Q_CREATIVE = "ตั้งชื่อสำนักงานกฎหมายใหม่ให้หน่อย 1 ชื่อ ตอบแค่ชื่อ"  # "Suggest one name for a new law firm. Name only."

for t in (0.0, 1.0):
    print(f"temperature = {t}")
    for i in range(3):
        print("  ", ask_local(Q_CREATIVE, temperature=t)[0].strip())

temperature = 0.0


   กฎหมายไทย
   กฎหมายไทย


   กฎหมายไทย
temperature = 1.0


   "กฎหมายรุ่งเรือง"


   ชื่อที่เหมาะสมสำหรับสำนักงานกฎหมายใหม่อาจเป็น "สำนักงานกฎหมายไทย"


   กฎหมายไทยกฎหมายไทย


## 4. Why LLMs hallucinate

An LLM is trained to produce **plausible next tokens**, not to look facts up. It has no database of laws inside — just patterns compressed into weights. When the true answer isn't well represented in its training data, the most *plausible-sounding* continuation is still produced, with the same confident tone. Analogy: a regression model will happily predict for an input far outside the training range — it never says "I don't know" unless you build that in.

Test: ask about a real section of the Thai Civil and Commercial Code (มาตรา 456, form of sale of immovable property) and one that **does not exist** (the Code ends at มาตรา 1755).

In [7]:
Q_REAL = "ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 456 บัญญัติว่าอย่างไร"
Q_FAKE = "ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1799 บัญญัติว่าอย่างไร"

show("Typhoon — real section 456", ask_local(Q_REAL))
show("Typhoon — non-existent section 1799", ask_local(Q_FAKE))
show("Gemini — non-existent section 1799", ask_gemini(Q_FAKE))

--- Typhoon — real section 456  (in=221 tokens, out=97 tokens)
ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 456 บัญญัติว่า "สัญญาเช่าซื้อจะสมบูรณ์ต่อเมื่อผู้ให้เช่าซื้อได้ส่งมอบทรัพย์สินที่เช่าซื้อแก่ผู้เช่าซื้อ และผู้เช่าซื้อได้ชำระเงินเช่าซื้อครั้งแรกแล้ว"



--- Typhoon — non-existent section 1799  (in=222 tokens, out=109 tokens)
ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1799 บัญญัติว่า "สัญญาเช่าซื้อจะสมบูรณ์ต่อเมื่อผู้ให้เช่าซื้อได้ส่งมอบทรัพย์สินที่เช่าซื้อแก่ผู้เช่าซื้อ และผู้เช่าซื้อได้ชำระเงินงวดแรกหรือได้ทำสัญญาเช่าซื้อเป็นหนังสือ"



--- Gemini — non-existent section 1799  (in=27 tokens, out=301 tokens)
**ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1799** บัญญัติไว้ว่า:

> **"ผู้จัดการมรดกไม่มีสิทธิที่จะได้รับบำเหน็จเนื่องในการปฏิบัติหน้าที่ เว้นแต่พินัยกรรมหรือความตกลงแห่งทายาทโดยเสียงข้างมากจะได้กำหนดไว้เป็นอย่างอื่น"**

---

### **คำอธิบายสาระสำคัญ:**
โดยหลักแล้ว การทำหน้าที่เป็นผู้จัดการมรดกถือเป็นหน้าที่ที่**ไม่มีค่าตอบแทน (บำเหน็จ)** แต่มีข้อเว้นที่ทำให้ผู้จัดการมรดกมีสิทธิได้รับค่าตอบแทนได้ 2 กรณี คือ:

1. **ระบุไว้ในพินัยกรรม:** เจ้าของมรดกเขียนไว้ในพินัยกรรมชัดเจนว่าให้บำเหน็จแก่ผู้จัดการมรดกเป็นจำนวนเท่าใด
2. **ทายาทเสียงข้างมากตกลงกัน:** ทายาทที่มีสิทธิในทรัพย์มรดกใช้เสียงข้างมากตกลงให้ค่าตอบแทนแก่ผู้จัดการมรดก

*(หมายเหตุ: บำเหน็จผู้จัดการมรดก แตกต่างจาก "ค่าใช้จ่ายจำเป็น" ในการจัดการมรดก เช่น ค่าธรรมเนียมศาล ค่าประกาศหนังสือพิมพ์ ซึ่งผู้จัดการมรดกสามารถหักเอาจากกองมรดกได้ตามจริง)*



**What to look for:** does either model invent text for มาตรา 1799? Is the 456 answer close to the real text ("การซื้อขายอสังหาริมทรัพย์ ถ้ามิได้ทำเป็นหนังสือและจดทะเบียนต่อพนักงานเจ้าหน้าที่ การซื้อขายนั้นเป็นโมฆะ…")?

This is the motivation for the whole project: **RAG** puts the actual law text into the prompt at answer time, so the model can quote and cite it instead of relying on memory — and our evaluation (M4, M7) measures how often that actually works.

## What we saw (run on 2026-11-03)

- **Tokens:** Thai ≈ 3.1 chars/token vs English ≈ 4.6 with Gemini's tokenizer → Thai costs ~1.5× more tokens for the same content. Typhoon's Llama-3.1 chat template silently adds a ~200-token default system prompt (221 input tokens for a one-line question).
- **System prompt:** Typhoon ignored "answer in the same language" (answered the English question in Thai); Gemini followed it.
- **Temperature:** 0 → identical answers; 1 → varied (and one degenerate repetition).
- **Hallucination:** Typhoon got มาตรา 456 wrong (described hire-purchase) and invented text for the non-existent มาตรา 1799. Gemini also invented มาตรา 1799 — a real-looking rule (estate administrator's remuneration) under a section number that doesn't exist, with confident formatting. **Neither model said "this section doesn't exist."** → motivation for RAG with checkable citations.
- **Ops:** `gemini-3.8-flash` timed out on the free tier; `gemini-3.5-flash` answers in ~2.5 s. Gemini 3.x uses hidden *thinking tokens* (68 to say "OK"). Ollama runs Typhoon 100 % on GPU but with an 8,192-token context by default.